In [11]:
import duckdb
con = duckdb.connect() 

In [2]:
con.read_csv("data/orders.csv").limit(2).show()

┌──────────┬────────────┬─────────────────────┬──────────────────┬───────────┬───────────┬────────────┬──────────┬────────────┬──────────────┬────────┬────────────────┬───────────┐
│ order_id │ order_date │     ordered_at      │     customer     │   city    │ category  │  product   │ quantity │ unit_price │ discount_pct │ amount │ payment_method │  status   │
│  int64   │    date    │      timestamp      │     varchar      │  varchar  │  varchar  │  varchar   │  int64   │   int64    │    int64     │ int64  │    varchar     │  varchar  │
├──────────┼────────────┼─────────────────────┼──────────────────┼───────────┼───────────┼────────────┼──────────┼────────────┼──────────────┼────────┼────────────────┼───────────┤
│        1 │ 2025-01-01 │ 2025-01-01 00:04:27 │ Nandini Malhotra │ Hyderabad │ Furniture │ Study Desk │        1 │       6999 │            5 │   6649 │ Credit Card    │ delivered │
│        2 │ 2025-01-01 │ 2025-01-01 01:24:10 │ Vikram Shah      │ Mumbai    │ Beauty    │ Suns

In [6]:
con.sql("FROM 'data/customers.json' LIMIT 2").show()
con.sql("SELECT customer, amount FROM 'data/orders.parquet' WHERE amount > 20000").show()


┌─────────────┬─────────────┬────────────────────────┬───────────┬─────────┬─────────────┐
│ customer_id │    name     │         email          │   city    │  tier   │ signup_date │
│    int64    │   varchar   │        varchar         │  varchar  │ varchar │    date     │
├─────────────┼─────────────┼────────────────────────┼───────────┼─────────┼─────────────┤
│           1 │ Karan Reddy │ karan.reddy@gmail.com  │ Bangalore │ premium │ 2023-07-09  │
│           2 │ Ananya Lal  │ ananya.lal@outlook.com │ Bangalore │ basic   │ 2024-12-28  │
└─────────────┴─────────────┴────────────────────────┴───────────┴─────────┴─────────────┘

┌─────────────────┬────────┐
│    customer     │ amount │
│     varchar     │ int32  │
├─────────────────┼────────┤
│ Arnav Mehta     │  59849 │
│ Rajiv Lal       │  56699 │
│ Naina Mehta     │  62999 │
│ Swati Sahni     │  62999 │
│ Shivani D'Souza │  62999 │
│ Anita Agarwal   │  62999 │
│ Ishaan Tiwari   │  32999 │
│ Reema Khan      │  59849 │
│ Rakesh Malho

In [7]:

con.sql("CREATE OR REPLACE TABLE orders_tbl AS FROM 'data/orders.csv'")   # create + fill
con.sql("INSERT INTO orders_tbl (order_id, customer, city, product, amount) VALUES (50001, 'Diya Shah', 'Pune', 'Sofa', 32999)")  # add rows; unlisted columns become NULL
con.sql("SELECT count(*) FROM orders_tbl").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        50001 │
└──────────────┘



In [8]:
con.sql("SHOW TABLES").show()          # orders (view), orders_tbl (table)

┌────────────┐
│    name    │
│  varchar   │
├────────────┤
│ orders_tbl │
└────────────┘



In [12]:
con.close()

In [13]:
con = duckdb.connect("shop.duckdb")       # file-backed: everything is saved to shop.duckdb   

In [14]:
con.sql("CREATE OR REPLACE VIEW orders AS FROM 'data/orders.parquet'")
con.sql("CREATE OR REPLACE TABLE orders_tbl AS FROM 'data/orders.csv'")
con.close()  

In [15]:
con = duckdb.connect("shop.duckdb")       # reconnect to the same file
con.sql("SHOW TABLES").show()    

┌────────────┐
│    name    │
│  varchar   │
├────────────┤
│ orders     │
│ orders_tbl │
└────────────┘



In [16]:
con.close()                               # release the file first: DuckDB won't ATTACH a file that
                                          # another connection in this process still has open
con = duckdb.connect()                    # back to in-memory
con.sql("ATTACH 'shop.duckdb' AS shop")   # open the file inside this connection
con.sql("SELECT count(*) FROM shop.orders_tbl").show()
con.sql("USE shop")   

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        50000 │
└──────────────┘



SuccessBOOLEAN


In [17]:
con.sql("""
SELECT city, count(*) AS orders, sum(amount) AS revenue
FROM orders
GROUP BY city
HAVING sum(amount) > 10000
ORDER BY revenue DESC
""").show()

┌────────────┬────────┬──────────┐
│    city    │ orders │ revenue  │
│  varchar   │ int64  │  int128  │
├────────────┼────────┼──────────┤
│ Mumbai     │   9191 │ 30760201 │
│ Bangalore  │   8617 │ 29392426 │
│ Delhi      │   8204 │ 25888403 │
│ Hyderabad  │   5024 │ 16892352 │
│ Chennai    │   4887 │ 16479809 │
│ Pune       │   4423 │ 14650337 │
│ Kolkata    │   3513 │ 11302329 │
│ Ahmedabad  │   2443 │  7565156 │
│ Jaipur     │   1328 │  4440832 │
│ Kochi      │    977 │  3279174 │
│ Lucknow    │    886 │  2626162 │
│ Chandigarh │    507 │  1789664 │
└────────────┴────────┴──────────┘
  12 rows              3 columns



In [18]:
con.sql("""
SELECT *
FROM duckdb_extensions()
""").show()


┌──────────────────┬─────────┬───────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────┬───────────────────────────┐
│  extension_name  │ loaded  │ installed │ install_path │                                    description                                     │      aliases      │ extension_version │   install_mode    │ installed_from │ signature_key_fingerprint │
│     varchar      │ boolean │  boolean  │   varchar    │                                      varchar                                       │     varchar[]     │      varchar      │      varchar      │    varchar     │          varchar          │
├──────────────────┼─────────┼───────────┼──────────────┼────────────────────────────────────────────────────────────────────────────────────┼───────────────────┼───────────────────┼───────────────────┼────────────────┼───────────────────────────┤
│ autoco